# Business Entity Resolution — Stage 2: Matcher → `matching_results.tsv`

```
candidate_pairs (from stage 1, pruned)  ──►  pairwise + context features  ──►  LightGBM P(match)
        ──►  per-S1 decision rule tuned for macro F0.5 (singletons included)  ──►  matching_results.tsv
```

**Inputs** (Kaggle → Add Data):
1. the competition dataset;
2. the **output of the candidate-generation notebook** run with `export_val_candidates=True`, which gives
   `val_candidate_pairs.parquet`, `val_queries.parquet`, `test_candidate_scores.parquet` and `candidate_config.json`.

**Why train on exported validation candidates?** They come from the *same* blocker and pruner as the test candidates,
so the matcher learns exactly the hard negatives it will face. The pruner never saw these queries.

**Decision rule (tuned on held-out validation queries, never on training queries):**
* keep candidate `c` of S1 `q` if `P(c) ≥ t`;
* **singleton gate:** output nothing for `q` if `max_c P(c) < g`. This matters because a false match on a
  singleton scores 0 and an empty prediction scores 1;
* optional stricter `t` for countries unseen in training (France).

The metric is the challenge's **macro F0.5 per S1**. S1s whose true matches were lost in blocking still count (their
recall is capped), and singletons score 1.0 only when the prediction is empty.

`matching_results.tsv` is a subset of `candidate_pairs.tsv` by construction: both are written from the same parquet.

In [ ]:
import os, re, csv, gc, json, time, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else "./working")
OUT_DIR = WORK_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = ["/kaggle/input", os.getcwd()]

def find_files(name):
    hits = []
    for root in SEARCH_ROOTS:
        if os.path.isdir(root):
            for dirpath, dirnames, filenames in os.walk(root):
                dirnames[:] = [d for d in dirnames if not d.startswith(".")]
                if name in filenames:
                    hits.append(os.path.join(dirpath, name))
    return sorted(set(hits))

def find_data_root(explicit=None):
    if explicit and os.path.isfile(os.path.join(explicit, "train", "train_source1.tsv")):
        return explicit
    hits = [h for h in find_files("train_source1.tsv") if os.path.basename(os.path.dirname(h)) == "train"]
    assert hits, "competition data (train/train_source1.tsv) not found - add the dataset or set DATA_ROOT"
    return os.path.dirname(os.path.dirname(hits[0]))

def test_path(data_root, i):
    for p in (os.path.join(data_root, "test", f"test_source{i}.tsv"), os.path.join(data_root, f"test_source{i}.tsv")):
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"test_source{i}.tsv")

def read_tsv(path, cols):
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False,
                     quoting=csv.QUOTE_NONE, usecols=cols)
    for c in cols:
        df[c] = df[c].astype(str).str.strip()
    return df[cols]

_PUNCT = r"[!-/:-@\[-`{-~¡-¿‐-‧‰-⁞　-〿।॥]"

def normalize_series(s):
    s = pd.Series(s).fillna("").astype(str)
    s = s.str.normalize("NFKC").str.lower()
    s = s.str.replace(_PUNCT, " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip().reset_index(drop=True)

def validate_id_list_file(path, header, required_ids, valid_s2=None, valid_s3=None):
    errors, seen, n_ids, n_empty = [], set(), 0, 0
    def err(m):
        if len(errors) < 30: errors.append(m)
    with open(path, encoding="utf-8") as f:
        h = f.readline().rstrip("\n").split("\t")
        if h != header: err(f"bad header {h}")
        for ln, line in enumerate(f, 2):
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 2: err(f"line {ln}: {len(parts)} columns"); continue
            s1id, ids = parts
            if s1id in seen: err(f"duplicate row {s1id}")
            seen.add(s1id)
            if s1id not in required_ids: err(f"unexpected S1 {s1id}")
            if ids == "": n_empty += 1; continue
            lst = ids.split(","); n_ids += len(lst)
            if len(set(lst)) != len(lst): err(f"{s1id}: duplicate ids")
            for x in lst:
                ok = (x.startswith("S2-") and (valid_s2 is None or x in valid_s2)) or \
                     (x.startswith("S3-") and (valid_s3 is None or x in valid_s3))
                if not ok: err(f"{s1id}: invalid id {x}")
    missing = required_ids - seen
    if missing: err(f"{len(missing):,} S1 rows missing")
    print(f"{os.path.basename(path)}: rows={len(seen):,} ids={n_ids:,} empty={n_empty:,} -> "
          + ("PASS" if not errors else f"FAIL ({len(errors)} issues)"))
    for e in errors[:10]: print("   -", e)
    return not errors

def write_id_list_file(path, header, s1_ids, groups):
    """groups: dict s1_id -> comma-joined string."""
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write("\t".join(header) + "\n")
        buf = []
        for s in s1_ids:
            buf.append(f"{s}\t{groups.get(s, '')}\n")
            if len(buf) >= 200_000: f.writelines(buf); buf = []
        f.writelines(buf)

In [ ]:
SEED = 42
DATA_ROOT = None
CONFIG = {
    "holdout_frac": 0.25,                 # validation S1 queries held out for early stopping + threshold tuning
    "lgb_params": dict(n_estimators=3000, learning_rate=0.05, num_leaves=127, min_child_samples=40,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0),
    "early_stopping_rounds": 100,
    "t_grid": [round(x, 3) for x in np.arange(0.05, 0.951, 0.025)],
    "gate_grid": [None] + [round(x, 2) for x in np.arange(0.10, 0.951, 0.05)],
    "unseen_t_boost": 0.05,               # France: t + boost (cannot be validated; slightly more precise)
    "feature_chunk": 2_000_000,
}

def pip_install(*p):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p])
if subprocess.run([sys.executable, "-c", "from rapidfuzz.process import cpdist"], capture_output=True).returncode:
    pip_install("rapidfuzz>=3.6")
try:
    import lightgbm as lgb
except ImportError:
    pip_install("lightgbm"); import lightgbm as lgb
from rapidfuzz import fuzz, process as rf_process
from rapidfuzz.distance import JaroWinkler
from tqdm.auto import tqdm

DATA_ROOT = find_data_root(DATA_ROOT)
hits = find_files("val_candidate_pairs.parquet")
assert hits, "val_candidate_pairs.parquet not found: re-run the candidate notebook with export_val_candidates=True"
RUN_DIR = os.path.dirname(hits[0])
for f in ("val_queries.parquet", "test_candidate_scores.parquet", "candidate_config.json"):
    assert os.path.isfile(os.path.join(RUN_DIR, f)), f"{f} missing in {RUN_DIR}"
CAND_CFG = json.load(open(os.path.join(RUN_DIR, "candidate_config.json")))
print("data:", DATA_ROOT, "\nstage-1 run:", RUN_DIR, "\nblocking model:", CAND_CFG.get("model_name"),
      "| pruner threshold:", (CAND_CFG.get("pruner") or {}).get("threshold"))

# optional cross-encoder scores from cross_encoder_rerank.ipynb (debug outputs are ignored)
CE_VAL, CE_TEST = find_files("ce_val_scores.parquet"), find_files("ce_test_scores.parquet")
USE_CE = bool(CE_VAL and CE_TEST)
print("cross-encoder features:", f"ON ({os.path.dirname(CE_VAL[0])})" if USE_CE else "OFF (ce_*_scores.parquet not attached)")

def attach_ce(df, path):
    ce = pd.read_parquet(path)
    key = ["source1_entity_id", "candidate_entity_id"]
    out = df.merge(ce.drop_duplicates(key), on=key, how="left", validate="many_to_one")
    assert len(out) == len(df)
    cov = out["ce_logit"].notna().mean()
    print(f"  {os.path.basename(path)}: coverage {cov:.2%}")
    assert cov > 0.99, "cross-encoder scores do not match these candidates (different stage-1 run?)"
    return out

## Features
Per record (computed once per unique record): normalised name / address (same normalisation as stage 1), name without
legal suffixes (`pvt ltd inc llc corp sarl sas …`), first name token, digit tokens, first number (house number),
postal code (last 5–6-digit number), Devanagari flag.

Per pair: blocking scores (`cosine`, `faiss_rank`, `prune_score`, exact-key flag, source), rapidfuzz name similarities
(`ratio`, `partial_ratio`, `token_sort`, `token_set`, Jaro-Winkler, and the same on suffix-stripped names), address
similarities, digit/house-number/postal agreement, missing-field flags, script mismatch, length differences.

Per S1 (context): number of candidates, rank of this candidate by `prune_score`, gap to the best `prune_score`,
cosine, name and address similarity among that S1's candidates. The model can then prefer the *best* candidate
instead of judging each pair in isolation.

In [ ]:
LEGAL_RE = (r"\b(?:pvt|private|ltd|limited|llc|llp|lp|plc|inc|incorporated|corp|corporation|co|company|"
            r"sarl|sas|sasu|sa|eurl|snc|sci|gmbh|the|and|et|of|m s|ms)\b")

def record_fields(name, address):
    n = normalize_series(name); a = normalize_series(address)
    ns = n.str.replace(LEGAL_RE, " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    digits = a.str.findall(r"\d+")
    return pd.DataFrame({
        "n": n.to_numpy(object), "ns": ns.to_numpy(object), "a": a.to_numpy(object),
        "first": ns.str.split(" ").str[0].fillna("").to_numpy(object),
        "d": digits.str.join(" ").to_numpy(object),
        "hn": digits.str[0].fillna("").to_numpy(object),
        "pc": a.str.findall(r"\b\d{5,6}\b").str[-1].fillna("").to_numpy(object),
        "deva": pd.Series(name).astype(str).str.contains(r"[ऀ-ॿ]", regex=True).to_numpy(),
    })

PAIR_FEATURES = ["cosine", "faiss_rank", "prune_score", "via_exact", "is_s3",
                 "n_ratio", "n_partial", "n_tsort", "n_tset", "n_jw", "ns_ratio", "ns_tset", "ns_equal", "n_equal",
                 "first_equal", "a_ratio", "a_tset", "a_partial", "d_tset", "hn_equal", "hn_both", "pc_equal",
                 "pc_both", "q_a_missing", "c_a_missing", "script_mismatch", "n_len_diff", "a_len_diff"]
CONTEXT_FEATURES = ["n_cands", "prune_rank", "prune_gap", "cos_gap", "n_tset_gap", "a_tset_gap"]
BLOCK_COLS = ["cosine", "faiss_rank", "prune_score", "via_exact", "is_s3"]
if USE_CE:
    PAIR_FEATURES.insert(5, "ce_logit"); BLOCK_COLS.append("ce_logit")
    CONTEXT_FEATURES += ["ce_rank", "ce_gap", "ce_minus_second"]
FEATURES = PAIR_FEATURES + CONTEXT_FEATURES

def _rf(a, b, scorer, scale=100.0):
    return rf_process.cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32) / scale

def pair_features(Q, C, blk):
    """Q, C: record_fields rows aligned with blk rows. blk: cosine, faiss_rank, prune_score, via_exact, is_s3."""
    F = np.full((len(blk), len(PAIR_FEATURES)), np.nan, dtype=np.float32)
    col = {f: i for i, f in enumerate(PAIR_FEATURES)}
    for f in BLOCK_COLS:
        F[:, col[f]] = blk[f].to_numpy(np.float32)
    qn, cn, qs, cs = Q["n"].tolist(), C["n"].tolist(), Q["ns"].tolist(), C["ns"].tolist()
    qa, ca, qd, cd = Q["a"].tolist(), C["a"].tolist(), Q["d"].tolist(), C["d"].tolist()
    F[:, col["n_ratio"]] = _rf(qn, cn, fuzz.ratio)
    F[:, col["n_partial"]] = _rf(qn, cn, fuzz.partial_ratio)
    F[:, col["n_tsort"]] = _rf(qn, cn, fuzz.token_sort_ratio)
    F[:, col["n_tset"]] = _rf(qn, cn, fuzz.token_set_ratio)
    F[:, col["n_jw"]] = _rf(qn, cn, JaroWinkler.normalized_similarity, 1.0)
    F[:, col["ns_ratio"]] = _rf(qs, cs, fuzz.ratio)
    F[:, col["ns_tset"]] = _rf(qs, cs, fuzz.token_set_ratio)
    qam, cam = (Q["a"].to_numpy() == ""), (C["a"].to_numpy() == "")
    for f, sc in (("a_ratio", fuzz.ratio), ("a_tset", fuzz.token_set_ratio), ("a_partial", fuzz.partial_ratio)):
        v = _rf(qa, ca, sc); v[qam | cam] = np.nan; F[:, col[f]] = v
    v = _rf(qd, cd, fuzz.token_set_ratio); v[(Q["d"].to_numpy() == "") | (C["d"].to_numpy() == "")] = np.nan
    F[:, col["d_tset"]] = v
    eq = lambda k: (Q[k].to_numpy() == C[k].to_numpy())
    both = lambda k: (Q[k].to_numpy() != "") & (C[k].to_numpy() != "")
    F[:, col["n_equal"]] = eq("n")
    F[:, col["ns_equal"]] = eq("ns") & (Q["ns"].to_numpy() != "")
    F[:, col["first_equal"]] = eq("first") & (Q["first"].to_numpy() != "")
    F[:, col["hn_both"]] = both("hn"); F[:, col["hn_equal"]] = np.where(both("hn"), eq("hn"), np.nan)
    F[:, col["pc_both"]] = both("pc"); F[:, col["pc_equal"]] = np.where(both("pc"), eq("pc"), np.nan)
    F[:, col["q_a_missing"]] = qam; F[:, col["c_a_missing"]] = cam
    F[:, col["script_mismatch"]] = Q["deva"].to_numpy() != C["deva"].to_numpy()
    ln = lambda L: np.fromiter(map(len, L), np.int32, len(L))
    F[:, col["n_len_diff"]] = np.abs(ln(qn) - ln(cn)); F[:, col["a_len_diff"]] = np.abs(ln(qa) - ln(ca))
    return F

def add_context(F, qcode):
    """Append per-S1 context features. qcode: integer S1 id per row."""
    col = {f: i for i, f in enumerate(PAIR_FEATURES)}
    d = pd.DataFrame({"q": qcode, "ps": np.nan_to_num(F[:, col["prune_score"]], nan=1.0),
                      "cos": F[:, col["cosine"]], "nt": F[:, col["n_tset"]], "at": F[:, col["a_tset"]]})
    g = d.groupby("q")
    ctx = np.column_stack([
        g["ps"].transform("size").to_numpy(np.float32),
        g["ps"].rank(ascending=False, method="min").to_numpy(np.float32),
        (d["ps"] - g["ps"].transform("max")).to_numpy(np.float32),
        (d["cos"] - g["cos"].transform("max")).to_numpy(np.float32),
        (d["nt"] - g["nt"].transform("max")).to_numpy(np.float32),
        (d["at"] - g["at"].transform("max")).to_numpy(np.float32),
    ])
    if USE_CE:
        d["ce"] = F[:, col["ce_logit"]]
        gc_ = d.groupby("q")["ce"]
        top1 = gc_.transform("max")
        rk = gc_.rank(ascending=False, method="first")
        second = d["q"].map(d.loc[rk == 2].set_index("q")["ce"]).to_numpy(np.float32)
        ctx = np.column_stack([ctx,
            gc_.rank(ascending=False, method="min").to_numpy(np.float32),
            (d["ce"] - top1).to_numpy(np.float32),
            np.where(d["ce"] >= top1, d["ce"] - second, d["ce"] - top1).astype(np.float32)])
    return np.hstack([F, ctx])

def block_frame(df):
    return pd.DataFrame({"cosine": np.where(df["faiss_rank"].to_numpy() < 0, np.nan, df["cosine"].to_numpy()),
                         "faiss_rank": np.where(df["faiss_rank"].to_numpy() < 0, np.nan, df["faiss_rank"].to_numpy()),
                         "prune_score": df["prune_score"].to_numpy(), "via_exact": df["via_exact"].to_numpy(),
                         "is_s3": (df["source"].astype(str).to_numpy() == "S3"),
                         **({"ce_logit": df["ce_logit"].to_numpy(np.float32)} if USE_CE else {})})

def features_for(df, Qf, q_idx, Cf, c_idx, qcode, chunk=CONFIG["feature_chunk"]):
    blk = block_frame(df)
    parts = []
    for a in tqdm(range(0, len(df), chunk), desc="features", disable=len(df) <= chunk):
        b = a + chunk
        parts.append(pair_features(Qf.iloc[q_idx[a:b]].reset_index(drop=True),
                                   Cf.iloc[c_idx[a:b]].reset_index(drop=True), blk.iloc[a:b]))
    return add_context(np.vstack(parts), qcode)

## Load labelled validation candidates and build features

In [ ]:
t = time.time()
vp = pd.read_parquet(os.path.join(RUN_DIR, "val_candidate_pairs.parquet"))
vq = pd.read_parquet(os.path.join(RUN_DIR, "val_queries.parquet"))
if USE_CE:
    vp = attach_ce(vp, CE_VAL[0])
print(f"validation: {len(vp):,} candidate pairs for {len(vq):,} S1 ({len(vp)/len(vq):.2f}/S1), "
      f"positives {vp['label'].mean():.2%}")
print(f"blocking recall ceiling on these S1: {vp['label'].sum() / vq['n_true'].sum():.4f}")

uq = vp.drop_duplicates("source1_entity_id")[["source1_entity_id", "s1_name", "s1_address"]].reset_index(drop=True)
uc = vp.drop_duplicates("candidate_entity_id")[["candidate_entity_id", "cand_name", "cand_address"]].reset_index(drop=True)
VQf = record_fields(uq["s1_name"], uq["s1_address"])
VCf = record_fields(uc["cand_name"], uc["cand_address"])
vq_idx = pd.Index(uq["source1_entity_id"]).get_indexer(vp["source1_entity_id"])
vc_idx = pd.Index(uc["candidate_entity_id"]).get_indexer(vp["candidate_entity_id"])
QID = pd.Index(vq["source1_entity_id"])
v_qcode = QID.get_indexer(vp["source1_entity_id"])
assert (v_qcode >= 0).all()
XV = features_for(vp, VQf, vq_idx, VCf, vc_idx, v_qcode)
YV = vp["label"].to_numpy(bool)
print(f"features {XV.shape} in {(time.time()-t)/60:.1f} min")

rng = np.random.default_rng(SEED)
hold_q = rng.random(len(vq)) < CONFIG["holdout_frac"]
hold_row = hold_q[v_qcode]
print(f"train S1 {(~hold_q).sum():,} ({(~hold_row).sum():,} pairs) | holdout S1 {hold_q.sum():,} ({hold_row.sum():,} pairs)")

## Train LightGBM (early stopping on held-out S1 queries)

In [ ]:
t = time.time()
model = lgb.LGBMClassifier(objective="binary", random_state=SEED, n_jobs=-1, verbose=-1, **CONFIG["lgb_params"])
model.fit(XV[~hold_row], YV[~hold_row], eval_set=[(XV[hold_row], YV[hold_row])], eval_metric="binary_logloss",
          callbacks=[lgb.early_stopping(CONFIG["early_stopping_rounds"], verbose=False), lgb.log_evaluation(200)])
BEST_ITER = model.best_iteration_ or CONFIG["lgb_params"]["n_estimators"]
p_hold = model.predict_proba(XV[hold_row], num_iteration=BEST_ITER)[:, 1]
from sklearn.metrics import roc_auc_score, average_precision_score
print(f"trained in {(time.time()-t)/60:.1f} min | best iteration {BEST_ITER} | holdout AUC "
      f"{roc_auc_score(YV[hold_row], p_hold):.4f} | AP {average_precision_score(YV[hold_row], p_hold):.4f}")
imp = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
print("top features (gain):", ", ".join(f"{k}" for k in imp.head(12).index))

## Tune the decision rule for macro F0.5 on the holdout S1
Per S1 `q` with `n_true` true matches (including those blocking missed):
* singleton (`n_true = 0`): F = 1 if nothing is predicted, else 0;
* otherwise: P = TP / #predicted, R = TP / n_true, F0.5 = 1.25·P·R / (0.25·P + R) (0 if nothing correct).

Reference points: **predict nothing**, **predict every candidate**, and the **oracle** (perfect matcher on these
candidates), which is the ceiling set by blocking.

In [ ]:
H_Q = np.flatnonzero(hold_q)                         # holdout S1 (positions in vq)
hq_pos = np.full(len(vq), -1); hq_pos[H_Q] = np.arange(len(H_Q))
hq = hq_pos[v_qcode[hold_row]]                       # holdout pair -> holdout S1 index
y_h = YV[hold_row]
nt_h = vq["n_true"].to_numpy()[H_Q]

def macro_f05(pred, q, y, nt):
    npred = np.bincount(q[pred], minlength=len(nt))
    tp = np.bincount(q[pred & y], minlength=len(nt))
    P = tp / np.maximum(npred, 1); R = tp / np.maximum(nt, 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        f = np.where(tp > 0, 1.25 * P * R / (0.25 * P + R), 0.0)
    F = np.where(nt == 0, (npred == 0).astype(float), f)
    sing = nt == 0
    return {"macro_f05": F.mean(), "f05_singletons": F[sing].mean() if sing.any() else np.nan,
            "f05_non_singletons": F[~sing].mean(), "precision_micro": tp.sum() / max(npred.sum(), 1),
            "recall_micro": tp.sum() / max(nt.sum(), 1), "avg_pred": npred.mean(),
            "singleton_empty_pct": (npred[sing] == 0).mean() * 100 if sing.any() else np.nan}

qmax_h = np.zeros(len(H_Q)); np.maximum.at(qmax_h, hq, p_hold)
refs = {"predict nothing": np.zeros(len(hq), bool), "predict all candidates": np.ones(len(hq), bool),
        "oracle (blocking ceiling)": y_h.copy()}
rows = [{"rule": k, **macro_f05(v, hq, y_h, nt_h)} for k, v in refs.items()]
best = None
for tt in CONFIG["t_grid"]:
    base = p_hold >= tt
    for g in CONFIG["gate_grid"]:
        if g is not None and g <= tt:
            continue
        pred = base if g is None else base & (qmax_h[hq] >= g)
        m = macro_f05(pred, hq, y_h, nt_h)
        if best is None or m["macro_f05"] > best[2]["macro_f05"]:
            best = (tt, g, m)
T_BEST, G_BEST, M_BEST = best
rows.append({"rule": f"LightGBM t={T_BEST} gate={G_BEST}", **M_BEST})
RULES_DF = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
    print(RULES_DF.to_string(index=False))
print(f"\nSelected: t={T_BEST}, gate={G_BEST} -> holdout macro F0.5 = {M_BEST['macro_f05']:.4f}")

# sensitivity around the optimum (robustness check)
sens = [(tt, macro_f05((p_hold >= tt) & (True if G_BEST is None else (qmax_h[hq] >= max(G_BEST, tt))), hq, y_h, nt_h)["macro_f05"])
        for tt in CONFIG["t_grid"]]
print("F0.5 vs t (gate fixed):", " ".join(f"{a:.2f}:{b:.4f}" for a, b in sens[::2]))
RULES_DF.to_csv(OUT_DIR / "matcher_validation_rules.csv", index=False)

## Refit on all validation candidates (train + holdout) with the early-stopped number of trees

In [ ]:
t = time.time()
params = dict(CONFIG["lgb_params"]); params["n_estimators"] = int(BEST_ITER * 1.1) + 1
final_model = lgb.LGBMClassifier(objective="binary", random_state=SEED, n_jobs=-1, verbose=-1, **params)
final_model.fit(XV, YV)
import pickle
pickle.dump({"model": final_model, "features": FEATURES, "t": T_BEST, "gate": G_BEST,
             "unseen_t_boost": CONFIG["unseen_t_boost"]}, open(OUT_DIR / "matcher.pkl", "wb"))
print(f"final model: {params['n_estimators']} trees in {(time.time()-t)/60:.1f} min")
del XV; gc.collect()

## Apply to test candidates → `matching_results.tsv` + `candidate_pairs.tsv`

In [ ]:
t = time.time()
tp_df = pd.read_parquet(os.path.join(RUN_DIR, "test_candidate_scores.parquet"),
                        columns=["source1_entity_id", "candidate_entity_id", "source", "cosine", "faiss_rank",
                                 "prune_score", "via_exact_full", "via_exact_name"])
tp_df["via_exact"] = tp_df["via_exact_full"] | tp_df["via_exact_name"]
if USE_CE:
    tp_df = attach_ce(tp_df, CE_TEST[0])
test_s1 = read_tsv(test_path(DATA_ROOT, 1), ["entity_id", "business_name", "business_address", "country"])
test_s2 = read_tsv(test_path(DATA_ROOT, 2), ["entity_id", "business_name", "business_address"])
test_s3 = read_tsv(test_path(DATA_ROOT, 3), ["entity_id", "business_name", "business_address"])
test_c = pd.concat([test_s2, test_s3], ignore_index=True)
print(f"test candidates {len(tp_df):,} for {tp_df['source1_entity_id'].nunique():,} S1 (of {len(test_s1):,})")

TQf = record_fields(test_s1["business_name"], test_s1["business_address"])
used = pd.Index(tp_df["candidate_entity_id"].unique())
c_rows = pd.Index(test_c["entity_id"]).get_indexer(used)
assert (c_rows >= 0).all(), "candidate ids missing from test S2/S3"
TCf = record_fields(test_c["business_name"].to_numpy()[c_rows], test_c["business_address"].to_numpy()[c_rows])
tq_idx = pd.Index(test_s1["entity_id"]).get_indexer(tp_df["source1_entity_id"])
tc_idx = used.get_indexer(tp_df["candidate_entity_id"])
XT = features_for(tp_df, TQf, tq_idx, TCf, tc_idx, tq_idx)
p_test = np.concatenate([final_model.predict_proba(XT[a:a + 2_000_000])[:, 1] for a in range(0, len(XT), 2_000_000)])
del XT; gc.collect()
print(f"test features + scoring: {(time.time()-t)/60:.1f} min")

train_countries = set(normalize_series(read_tsv(os.path.join(DATA_ROOT, "train", "train_source1.tsv"), ["country"])["country"].drop_duplicates()))
unseen_q = ~normalize_series(test_s1["country"]).isin(train_countries).to_numpy()
t_row = np.where(unseen_q[tq_idx], T_BEST + CONFIG["unseen_t_boost"], T_BEST)
qmax_t = np.zeros(len(test_s1)); np.maximum.at(qmax_t, tq_idx, p_test)
pred = p_test >= t_row
if G_BEST is not None:
    g_row = np.where(unseen_q[tq_idx], G_BEST + CONFIG["unseen_t_boost"], G_BEST)
    pred &= qmax_t[tq_idx] >= g_row

tp_df["p"] = p_test
tp_df = tp_df.sort_values(["source1_entity_id", "p"], ascending=[True, False], kind="stable")
cand_groups = tp_df.groupby("source1_entity_id", sort=False)["candidate_entity_id"].agg(",".join).to_dict()
keep_df = tp_df[pred[tp_df.index.to_numpy()]]
match_groups = keep_df.groupby("source1_entity_id", sort=False)["candidate_entity_id"].agg(",".join).to_dict()

S1_IDS = test_s1["entity_id"].tolist()
MATCH_PATH, CAND_PATH = OUT_DIR / "matching_results.tsv", OUT_DIR / "candidate_pairs.tsv"
write_id_list_file(MATCH_PATH, ["source1_entity_id", "matched_entity_ids"], S1_IDS, match_groups)
write_id_list_file(CAND_PATH, ["source1_entity_id", "candidate_entity_ids"], S1_IDS, cand_groups)
n_match = np.bincount(tq_idx[pred], minlength=len(test_s1))
print(f"predicted matches: {pred.sum():,} | per S1 {n_match.mean():.2f} | empty S1 {(n_match == 0).mean():.1%}")
print(pd.DataFrame({"country": test_s1["country"], "n": n_match}).groupby("country")["n"].agg(["mean", lambda s: (s == 0).mean()])
      .rename(columns={"<lambda_0>": "empty_frac"}).to_string())

## Validate both files (own checks + official `utils/validate_submission.py`)

In [ ]:
valid_s2, valid_s3 = set(test_s2["entity_id"]), set(test_s3["entity_id"])
req = set(S1_IDS)
ok_m = validate_id_list_file(MATCH_PATH, ["source1_entity_id", "matched_entity_ids"], req, valid_s2, valid_s3)
ok_c = validate_id_list_file(CAND_PATH, ["source1_entity_id", "candidate_entity_ids"], req, valid_s2, valid_s3)
subset_ok = all(set(v.split(",")) <= set(cand_groups.get(k, "").split(",")) for k, v in match_groups.items())
print("matches subset of candidates:", subset_ok)

val_script = next((h for h in find_files("validate_submission.py")), None)
if val_script:
    test_dir = os.path.dirname(test_path(DATA_ROOT, 1))
    r = subprocess.run([sys.executable, val_script, "--matching", str(MATCH_PATH), "--candidate", str(CAND_PATH),
                        "--test-dir", test_dir], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-1000:])
else:
    print("utils/validate_submission.py not found in inputs - run it locally before submitting.")

json.dump({"t": T_BEST, "gate": G_BEST, "unseen_t_boost": CONFIG["unseen_t_boost"], "best_iteration": int(BEST_ITER),
           "holdout_metrics": M_BEST, "features": FEATURES, "stage1_run": RUN_DIR, "cross_encoder": USE_CE},
          open(OUT_DIR / "matcher_config.json", "w"), indent=1, default=float)
print(f"""
==================================================
MATCHER SUMMARY
==================================================
holdout macro F0.5      : {M_BEST['macro_f05']:.4f}  (singletons {M_BEST['f05_singletons']:.4f}, others {M_BEST['f05_non_singletons']:.4f})
oracle on candidates    : {RULES_DF.loc[RULES_DF.rule.str.startswith('oracle'), 'macro_f05'].iloc[0]:.4f}
decision rule           : P >= {T_BEST}, gate {G_BEST} (unseen countries +{CONFIG['unseen_t_boost']})
test matches / S1       : {n_match.mean():.2f}   empty S1: {(n_match == 0).mean():.1%}
files valid             : matching {ok_m}, candidates {ok_c}, subset {subset_ok}
outputs                 : {MATCH_PATH}
                          {CAND_PATH}
==================================================""")